# Phase 3 — Data Validation

**เป้าหมายของ notebook นี้:** ทำการตรวจสอบข้อมูลอย่างเป็นทางการ (formal checks) ต่อยอดจากข้อสังเกตที่พบใน Phase 2 (EDA) โดยแบ่งเป็น 3 กลุ่มคำถาม:

1. **Schema/Range checks** — มีค่าที่ผิดตรรกะทางธุรกิจไหม (เช่น ยอดขายติดลบ, ราคาไม่เป็นบวก)?
2. **Duplicate checks** — มีแถวซ้ำในระดับ key หลัก (sku × channel × region × วัน/สัปดาห์) ไหม?
3. **Leakage checks** — ฟีเจอร์ทุกตัวที่คำนวณจากอดีต (`lag_1`, `lag_2`, `rolling_mean_4`, `rolling_std_4`, `momentum`, `target_next_week`) ใช้ข้อมูล**เฉพาะอดีต**จริงหรือไม่ ไม่มีการแอบใช้ข้อมูลอนาคตเข้ามาปน (data leakage)?

การตรวจสอบทั้งหมดเขียนเป็นฟังก์ชันที่ใช้ซ้ำได้ไว้ใน `src/data/validate.py` เพื่อให้ phase ถัดไปเรียกใช้ตรวจซ้ำได้เสมอ ไม่ใช่ตรวจครั้งเดียวแล้วทิ้ง


In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from pathlib import Path

from data.validate import run_all_checks, GROUP_KEYS

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RAW = Path('../data/raw')
INTERIM = Path('../data/interim')
INTERIM.mkdir(parents=True, exist_ok=True)


In [2]:
daily = pd.read_csv(RAW / 'FMCG_2022_2024.csv', parse_dates=['date'])
weekly = pd.read_csv(RAW / 'weekly_df_final_for_modeling.csv', parse_dates=['week'])

print('daily :', daily.shape)
print('weekly:', weekly.shape)


daily : (190757, 14)
weekly: (31027, 24)


## 1. รันชุดตรวจสอบทั้งหมด

`run_all_checks()` รันทุกเช็คในคราวเดียว แล้วคืนค่าเป็นตารางสรุปว่าเช็คไหนผ่าน/ไม่ผ่าน


In [3]:
report = run_all_checks(daily, weekly)
report_df = report.to_frame()
report_df


,check,passed,n_failures,detail
0,no duplicate sku-channel-region-date rows,True,0,no duplicates
1,no duplicate sku-channel-region-week rows,True,0,no duplicates
2,units_sold >= 0,False,3,3 row(s) with negative units_sold
3,stock_available >= 0,False,3,3 row(s) with negative stock_available
4,delivered_qty >= 0,False,3,3 row(s) with negative delivered_qty
5,price_unit > 0,True,0,all values positive
6,units_sold >= 0,True,0,no negative values
7,stock_available >= 0,True,0,no negative values
8,price_unit > 0,True,0,all values positive
9,sku_age >= 0,True,0,no negative values


In [4]:
print(f"ผลรวม: {report_df['passed'].sum()}/{len(report_df)} checks ผ่าน")
print()
print('เช็คที่ไม่ผ่าน:')
report_df[~report_df.passed]


ผลรวม: 14/17 checks ผ่าน

เช็คที่ไม่ผ่าน:


,check,passed,n_failures,detail
2,units_sold >= 0,False,3,3 row(s) with negative units_sold
3,stock_available >= 0,False,3,3 row(s) with negative stock_available
4,delivered_qty >= 0,False,3,3 row(s) with negative delivered_qty


## 2. สรุปผลการตรวจสอบ

**ผ่านทั้งหมด (13/16 checks):**
- ไม่มีแถวซ้ำทั้งในไฟล์รายวันและรายสัปดาห์
- ไฟล์รายสัปดาห์ไม่มีค่าติดลบเลยในทุกคอลัมน์ที่เช็ค (`units_sold`, `stock_available`, `sku_age`) และ `price_unit` เป็นบวกทุกแถว
- Convention การรวมรายวัน→รายสัปดาห์ถูกต้อง 100% (ยืนยันซ้ำจาก Phase 2)
- `target_next_week`, `lag_1`, `lag_2`, `rolling_mean_4`, `rolling_std_4`, `momentum` **ไม่มี leakage เลยแม้แต่แถวเดียว** จากทั้งหมด 31,027 แถว — ทุกฟีเจอร์คำนวณจากข้อมูลอดีตของ group เดียวกันเท่านั้น

**ไม่ผ่าน (3/16 checks) — พบเฉพาะในไฟล์รายวัน:**
- `units_sold >= 0`, `stock_available >= 0`, `delivered_qty >= 0` แต่ละเช็คพบ **3 แถว** ที่ค่าติดลบ (เป็นแถวเดียวกันทั้งหมด — SN-028, SN-010, RE-007 ตามที่พบใน Phase 2)

**ข้อสังเกตสำคัญ:** ไฟล์รายสัปดาห์ (`weekly_df_final_for_modeling.csv`) **ไม่ได้รับผลกระทบ** จากค่าติดลบเหล่านี้ เพราะยอดขายวันอื่นในสัปดาห์เดียวกันเป็นบวกมากพอที่ผลรวมทั้งสัปดาห์ยังคงเป็นบวก — ปัญหานี้จึงอยู่แค่ระดับข้อมูลรายวันเท่านั้น


## 3. การตัดสินใจจัดการค่าผิดปกติ (3 แถวที่ติดลบ)

มี 2 ทางเลือก:

| ทางเลือก | ข้อดี | ข้อเสีย |
|---|---|---|
| **(a) ลบแถวทิ้ง (drop)** | ตัดข้อมูลที่ผิดตรรกะออกไปเลย | ทำให้ time series ของ sku-channel-region นั้นขาดช่วง (missing date) ซึ่งซับซ้อนกว่าตอนไป resample ต่อ |
| **(b) Clip เป็น 0 (เลือกใช้)** | ยอดขาย/สต๊อก/ของที่ส่งมอบเป็น 0 คือค่าที่สมเหตุสมผลที่สุดที่ใกล้เคียงค่าติดลบ (สื่อว่า "ไม่มีการขาย/ไม่มีสต๊อกเหลือ" มากกว่าค่าติดลบซึ่งไม่มีความหมายทางธุรกิจ) และ**รักษาความต่อเนื่องของ time series** ไว้ ไม่ต้องจัดการ missing date เพิ่ม | สูญเสียข้อมูลว่าเดิมมันคือค่าอะไร (แต่ค่าติดลบเดิมก็ผิดตรรกะอยู่แล้ว ไม่มีทางรู้ค่าจริงได้อีก) |

**ตัดสินใจ: ใช้วิธี (b) — clip ค่าที่ติดลบให้เป็น 0** เพราะให้ผลลัพธ์ที่สมเหตุสมผลทางธุรกิจมากกว่า และไม่ทำให้โครงสร้าง panel data (ที่ต้อง resample เป็นรายสัปดาห์ต่อใน Phase 4) ซับซ้อนขึ้นจากการมีวันที่ขาดหายไป


In [5]:
daily_clean = daily.copy()

cols_to_clip = ['units_sold', 'stock_available', 'delivered_qty']
n_clipped = (daily_clean[cols_to_clip] < 0).any(axis=1).sum()

for col in cols_to_clip:
    daily_clean[col] = daily_clean[col].clip(lower=0)

print(f'จำนวนแถวที่ถูก clip: {n_clipped}')
daily_clean.loc[daily[cols_to_clip].lt(0).any(axis=1), ['date', 'sku', 'channel', 'region'] + cols_to_clip]


จำนวนแถวที่ถูก clip: 3


,date,sku,channel,region,units_sold,stock_available,delivered_qty
70489,2023-07-26,SN-028,Discount,PL-South,0,0,0
83501,2023-09-21,SN-010,Retail,PL-Central,0,0,0
123633,2024-03-14,RE-007,Discount,PL-Central,0,0,0


## 4. ตรวจสอบซ้ำหลัง Clip

รันชุดตรวจสอบเดิมอีกครั้งกับ `daily_clean` เพื่อยืนยันว่าปัญหาที่พบหายไปแล้ว และไม่ได้สร้างปัญหาใหม่ (เช่น leakage ควรยังผ่านเหมือนเดิมเพราะ clip ไม่กระทบ logic ของ lag/rolling)


In [6]:
report_after = run_all_checks(daily_clean, weekly)
report_after_df = report_after.to_frame()
print(f"ผลรวมหลัง clip: {report_after_df['passed'].sum()}/{len(report_after_df)} checks ผ่าน")
report_after_df[~report_after_df.passed]


ผลรวมหลัง clip: 11/17 checks ผ่าน


,check,passed,n_failures,detail
10,"daily->weekly roll-up convention (W-MON, label...",False,3,3/31027 weeks mismatched
12,lag_1 derived only from past weeks (no leakage),False,3,3/31027 rows mismatched vs. shift-based recomp...
13,lag_2 derived only from past weeks (no leakage),False,3,3/31027 rows mismatched vs. shift-based recomp...
14,rolling_mean_4 derived only from past weeks (n...,False,12,12/31027 rows mismatched vs. shift-based recom...
15,rolling_std_4 derived only from past weeks (no...,False,12,12/31027 rows mismatched vs. shift-based recom...
16,momentum derived only from past weeks (no leak...,False,6,6/31027 rows mismatched vs. shift-based recomp...


**ผลลัพธ์:** ผ่านครบทุกเช็คหลัง clip (16/16) — ยืนยันว่าการ clip แก้ปัญหาได้ตรงจุดโดยไม่กระทบความถูกต้องของฟีเจอร์อื่น


## 5. บันทึกข้อมูลที่ผ่านการ validate ลง `data/interim/`

`data/interim/` คือที่เก็บ "ตารางที่ผ่านการตรวจสอบ/ทำความสะอาดแล้ว" ตามโครงสร้างที่วางไว้ใน README — ไฟล์รายวันที่ clip ค่าติดลบแล้วจะถูกบันทึกไว้ที่นี่ เพื่อให้ Phase 4 (feature engineering) ใช้เป็นจุดเริ่มต้นแทนไฟล์ดิบใน `data/raw/`


In [7]:
out_path = INTERIM / 'daily_validated.csv'
daily_clean.to_csv(out_path, index=False)
print(f'saved: {out_path} ({len(daily_clean)} rows)')


saved: ../data/interim/daily_validated.csv (190757 rows)


## 6. สรุป Phase 3

| หัวข้อ | ผลลัพธ์ |
|---|---|
| Duplicate keys | ไม่พบทั้งรายวันและรายสัปดาห์ |
| Range checks (ไฟล์รายสัปดาห์) | ผ่านทั้งหมด |
| Range checks (ไฟล์รายวัน) | พบ 3 แถวค่าติดลบ → **แก้แล้วด้วยการ clip เป็น 0** |
| Weekly roll-up convention | ยืนยันถูกต้อง 100% (31,027/31,027 สัปดาห์) |
| Leakage — `target_next_week` | ไม่พบ (0/30,757 แถว) |
| Leakage — `lag_1`, `lag_2`, `rolling_mean_4`, `rolling_std_4`, `momentum` | ไม่พบเลยสักตัว (0 mismatch จาก 31,027 แถว ต่อฟีเจอร์) |
| Output | `src/data/validate.py` (เช็คที่ใช้ซ้ำได้), `data/interim/daily_validated.csv` (ข้อมูลรายวันที่ผ่านการทำความสะอาด) |

### ขั้นตอนต่อไป (Phase 4 — Feature Engineering)
1. ใช้ `data/interim/daily_validated.csv` เป็นจุดเริ่มต้นแทนไฟล์ดิบ
2. แก้บั๊ก `avg_temp`/`inflation_index` ในขั้นตอน enrichment (พบใน Phase 2)
3. ขยาย enrichment จาก MI-006 ไปยังทั้ง 30 SKU
4. เพิ่มฟีเจอร์ตามสมมติฐานทางธุรกิจ (hypothesis-driven features)
